# 🇪🇺 EUR-Lex 법령 챗봇 (Google Colab)

EU 공식 법령 데이터베이스(EUR-Lex / Cellar)의 공개 API와 OpenAI GPT를 연결한 챗봇입니다.

**동작 방식**
1. 사용자가 질문하면 GPT가 필요한 도구를 스스로 고릅니다(function calling).
2. `search_eurlex` — SPARQL 엔드포인트로 법령을 검색 (CELEX 번호, 제목, 날짜)
3. `get_document` — CELEX 번호로 본문을 가져옴 (특정 조항만 뽑기 가능)
4. GPT가 원문을 근거로 한국어로 답하고 출처 링크를 붙입니다.

**준비물**: OpenAI API 키 → 왼쪽 사이드바 🔑(보안 비밀)에 이름 `OPENAI_API_KEY`로 저장하고 "노트북 액세스"를 켜 주세요.

위에서부터 셀을 차례로 실행하면 됩니다.

## 🔧 고친 점 (「분석 한도에 도달했다」 답이 나오던 문제)

**원인**: 「⚠️ 도구 호출 횟수 한도에 도달했습니다」는 `ask()` 가 도구를 8번 부르고도 답을 못 받았을 때 내는 말입니다. 실제로 재어 보니
- EUR-Lex 제목 검색(SPARQL) 한 번에 **10~28초**가 걸리고, 「high-risk AI system」처럼 키워드가 많으면 **결과 0건**이 나옵니다.
- 그러면 GPT 가 키워드를 바꿔 다시 찾기를 되풀이하다 8번을 다 쓰고, 그 자리에서 답 없이 끝났습니다. 그래서 「한참 기다린 뒤 한도 도달」로 보였습니다.

**고친 것**
| | 전 | 후 |
|---|---|---|
| 도구 횟수를 다 썼을 때 | 답 없이 「한도 도달」 | 도구를 끄고(`tool_choice="none"`) **모은 원문으로 답을 받아 냄** |
| 자주 묻는 법령(GDPR · AI Act · DSA · DMA 등 8개) | 매번 검색(10~28초) | 검색 없이 CELEX 로 바로 본문(약 2초) |
| 같은 도구 · 같은 값 반복 | 매번 다시 실행 | 「이미 불렀다」고 알려 줌 |
| 조항을 못 찾으면 | 본문 앞 15,000자를 보냄 | **목차(Article 번호 · 제목)** 를 돌려줘 바로 고르게 함 |
| 검색어의 `AI` | 3글자 미만이라 빠짐 | `artificial intelligence` 로 풀어 씀 |
| gpt-5-mini 생각 시간 | 기본(길다) | `reasoning_effort="low"` |
| 채팅 화면 | 다 끝날 때까지 빈칸 | 지금 무슨 도구를 쓰는지 바로바로 보여 줌 |
| 대화 기록 | `[질문, 답]` 짝으로 풀기(지금 Gradio 6 형식과 달라 두 번째 질문부터 꼬임) | `{"role", "content"}` 형식으로 읽음 |

질문할 때 **법령 이름 + 조항 번호**(예: 「AI Act 6조」)를 함께 적으면 가장 빨리 답합니다.

## 1. 패키지 설치

In [ ]:
!pip install -q -U openai gradio beautifulsoup4


## 2. API 키 불러오기

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"   # 계정에서 쓸 수 있는 다른 모델(예: "gpt-5", "gpt-4.1")로 바꿔도 됩니다

def model_options():
    # gpt-5 · o 계열(추론 모델)은 생각을 오래 하면 답이 늦어집니다 → 짧게 생각하도록 합니다
    return {"reasoning_effort": "low"} if MODEL.startswith(("gpt-5", "o3", "o4")) else {}

print("✅ 준비 완료 ·", MODEL)


## 3. EUR-Lex 검색 도구 (SPARQL)

`publications.europa.eu`의 공개 SPARQL 엔드포인트를 사용합니다. 가입이나 키가 필요 없습니다.
제목 키워드 + 문서 종류 + 연도로 거르고, 결과 수를 제한해 타임아웃을 피합니다.

In [ ]:
import requests, re
from functools import lru_cache

SPARQL_URL = "https://publications.europa.eu/webapi/rdf/sparql"
RT = "http://publications.europa.eu/resource/authority/resource-type/"

DOC_TYPES = {            # 사람이 쓰는 이름 → Cellar 리소스 타입 코드
    "regulation": "REG",
    "directive": "DIR",
    "decision": "DEC",
    "implementing_regulation": "REG_IMPL",
    "delegated_regulation": "REG_DEL",
    "judgment": "JUDG",
}

# 자주 묻는 법령은 검색하지 않고 바로 CELEX 로 찾습니다 (검색이 가장 느린 단계라서)
KNOWN = {
    "GDPR(개인정보 보호)": "32016R0679",
    "AI Act(인공지능법)": "32024R1689",
    "DSA(디지털서비스법)": "32022R2065",
    "DMA(디지털시장법)": "32022R1925",
    "Data Act(데이터법)": "32023R2854",
    "Battery Regulation(배터리 규정)": "32023R1542",
    "CBAM(탄소국경조정)": "32023R0956",
    "NIS2 Directive(사이버보안)": "32022L2555",
}

@lru_cache(maxsize=64)
def _sparql(query):
    r = requests.get(SPARQL_URL,
                     params={"query": query, "format": "application/sparql-results+json"},
                     timeout=40)
    r.raise_for_status()
    return r.json()["results"]["bindings"]

def search_eurlex(keywords: str, doc_type: str = None, year_from: int = None, limit: int = 10):
    """제목에 키워드가 모두 들어간 EU 법령을 최신순으로 검색"""
    # 'AI' 처럼 두 글자 약어는 아래 규칙(3글자 이상)에서 빠지므로 풀어 씁니다
    keywords = re.sub(r"\bAI\b", "artificial intelligence", keywords, flags=re.I)
    words = [w.lower() for w in re.findall(r"[\w-]+", keywords) if len(w) > 2][:4]
    if not words:
        return {"error": "검색어가 비어 있습니다."}
    filters = "\n".join(
        f'  FILTER(CONTAINS(LCASE(STR(?title)), "{w.replace(chr(34), "")}"))' for w in words
    )
    type_clause = ""
    if doc_type and doc_type in DOC_TYPES:
        type_clause = f"?work cdm:work_has_resource-type <{RT}{DOC_TYPES[doc_type]}> ."
    year_clause = f'FILTER(YEAR(?date) >= {int(year_from)})' if year_from else ""

    query = f"""
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
SELECT DISTINCT ?celex ?date ?title WHERE {{
  ?work cdm:resource_legal_id_celex ?celex ;
        cdm:work_date_document ?date .
  {type_clause}
  ?expr cdm:expression_belongs_to_work ?work ;
        cdm:expression_title ?title ;
        cdm:expression_uses_language <http://publications.europa.eu/resource/authority/language/ENG> .
{filters}
  {year_clause}
}}
ORDER BY DESC(?date)
LIMIT {min(int(limit or 10), 15)}
"""
    try:
        rows = _sparql(query)
    except requests.Timeout:
        return {"error": "검색이 40초 안에 끝나지 않았습니다. 키워드를 줄이거나 year_from 을 넣어 한 번만 다시 찾으세요.",
                "known_celex": KNOWN}
    except Exception as e:
        return {"error": f"SPARQL 요청 실패: {e}", "known_celex": KNOWN}

    results, seen = [], set()
    for row in rows:
        celex = row["celex"]["value"]
        if celex in seen:
            continue
        seen.add(celex)
        results.append({
            "celex": celex,
            "date": row["date"]["value"][:10],
            "title": row["title"]["value"][:300],
            "url": f"https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:{celex}",
        })
    if not results:
        return {"count": 0, "results": [], "hint": "제목에 모든 키워드가 들어간 문서가 없습니다. 키워드를 1~2개로 줄여 보세요."}
    return {"count": len(results), "results": results}

# 테스트
search_eurlex("artificial intelligence", doc_type="regulation", year_from=2020, limit=5)


## 4. 본문 가져오기 도구

CELEX 번호로 Cellar REST API에서 HTML 본문을 받아 텍스트로 정리합니다(실패하면 EUR-Lex 페이지로 재시도).
`article`을 주면 해당 조항만 잘라 반환해서 토큰을 아낍니다.

In [ ]:
from bs4 import BeautifulSoup

LANG3 = {"EN": "eng", "DE": "deu", "FR": "fra", "ES": "spa", "IT": "ita"}

@lru_cache(maxsize=64)
def _fetch_text(celex: str, lang: str = "EN") -> str:
    headers = {"Accept": "text/html, application/xhtml+xml",
               "Accept-Language": LANG3.get(lang, "eng"),
               "User-Agent": "Mozilla/5.0 (EUR-Lex chatbot demo)"}
    urls = [
        f"http://publications.europa.eu/resource/celex/{celex}",                       # Cellar REST
        f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/HTML/?uri=CELEX:{celex}",  # 대체 경로
    ]
    last_err = None
    for url in urls:
        try:
            r = requests.get(url, headers=headers, timeout=60, allow_redirects=True)
            if r.status_code == 300:   # 여러 형식이 있을 때: 목록 중 HTML 링크를 따라감
                links = re.findall(r'href="([^"]+)"', r.text)
                html_links = [l for l in links if "html" in l.lower()] or links
                if html_links:
                    r = requests.get(html_links[0], headers=headers, timeout=60)
            r.raise_for_status()
            soup = BeautifulSoup(r.content, "html.parser")
            for t in soup(["script", "style", "nav", "header", "footer"]):
                t.decompose()
            text = re.sub(r"\n{3,}", "\n\n", soup.get_text("\n")).strip()
            if len(text) > 500:
                return text
        except Exception as e:
            last_err = e
    raise RuntimeError(f"본문을 가져오지 못했습니다: {last_err}")

def list_articles(text, limit=120):
    """본문에서 「Article 번호 + 제목」 목차만 뽑습니다"""
    heads = re.findall(r"\n\s*Article\s+(\d+[a-z]?)\s*\n\s*([^\n]{3,120})", text)
    seen, out = set(), []
    for num, title in heads:
        if num not in seen:
            seen.add(num)
            out.append(f"Article {num} — {title.strip()}")
    return out[:limit]

def get_document(celex: str, article: str = None, language: str = "EN", max_chars: int = 8000):
    """CELEX 번호로 법령 본문(또는 특정 조항)을 반환"""
    try:
        text = _fetch_text(celex.strip(), (language or "EN").upper())
    except Exception as e:
        return {"error": str(e)}
    url = f"https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:{celex}"

    if article:
        num = re.sub(r"[^\da-z]", "", str(article).lower())
        m = re.search(rf"\n\s*Article\s+{num}\s*\n(.*?)(?=\n\s*Article\s+\d+[a-z]?\s*\n|\Z)", text, re.S)
        if not m:
            # 조항을 못 찾으면 본문 앞부분(수천 자) 대신 목차를 돌려줘서 다음 호출에서 바로 고르게 합니다
            return {"celex": celex, "url": url,
                    "error": f"Article {num} 을(를) 찾지 못했습니다. 아래 목차에서 번호를 골라 다시 부르세요.",
                    "articles": list_articles(text)}
        text = f"Article {num}\n" + m.group(1).strip()
    else:
        # 조항을 정하지 않으면: 앞부분(제목 · 전문) + 목차
        toc = "\n".join(list_articles(text))
        text = text[:3000] + "\n\n[목차]\n" + toc

    return {"celex": celex, "url": url, "truncated": len(text) > max_chars, "text": text[:max_chars]}

# 테스트: GDPR 제17조(잊힐 권리)
print(get_document("32016R0679", article="17")["text"][:1500])


## 5. GPT에 도구 연결 (Function calling)

In [ ]:
import json

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_eurlex",
            "description": ("EUR-Lex에서 EU 법령을 제목 키워드로 검색한다. 키워드는 반드시 영어로, 1~3 낱말만 넣는다. "
                            "CELEX 번호, 날짜, 제목, URL 목록을 반환한다. 아래 자주 쓰는 법령은 검색하지 말고 바로 get_document 를 쓴다."),
            "parameters": {
                "type": "object",
                "properties": {
                    "keywords": {"type": "string", "description": "영어 키워드 1~3 낱말 (예: 'battery')"},
                    "doc_type": {"type": "string", "enum": list(DOC_TYPES.keys())},
                    "year_from": {"type": "integer", "description": "이 연도 이후 문서만"},
                    "limit": {"type": "integer", "description": "최대 결과 수 (기본 10)"},
                },
                "required": ["keywords"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_document",
            "description": ("CELEX 번호로 EU 법령 원문을 가져온다. 특정 조항이 필요하면 article에 조항 번호를 넣는다. "
                            "조항 번호를 모르면 article 없이 불러 목차를 먼저 본다."),
            "parameters": {
                "type": "object",
                "properties": {
                    "celex": {"type": "string"},
                    "article": {"type": "string", "description": "조항 번호 (예: '17')"},
                    "language": {"type": "string", "enum": list(LANG3.keys())},
                },
                "required": ["celex"],
            },
        },
    },
]

FUNCS = {"search_eurlex": search_eurlex, "get_document": get_document}

KNOWN_TEXT = "\n".join(f"  - {k}: {v}" for k, v in KNOWN.items())
SYSTEM = f"""너는 EU 법령 전문 리서치 어시스턴트다.
- 반드시 도구로 EUR-Lex 원문을 확인한 뒤 답한다. 기억에만 의존해 조항 내용을 단정하지 않는다.
- 도구는 질문 하나에 많아야 4번까지 쓴다. 같은 도구를 같은 값으로 다시 부르지 않는다.
- 검색 결과가 없거나 실패하면 키워드를 바꿔 한 번만 다시 찾고, 그래도 없으면 지금까지 확인한 것으로 답한다.
- 자주 쓰는 법령은 검색하지 말고 바로 get_document 를 쓴다:
{KNOWN_TEXT}
- 한국어로 답하되, 핵심 조문은 짧게 영어 원문을 인용하고 번역을 붙인다.
- 답변 끝에 근거 문서의 CELEX 번호, 조항, EUR-Lex 링크를 '출처'로 정리한다.
- 원문에서 확인되지 않는 내용은 확인되지 않았다고 말한다.
- 법률 자문이 아니라 정보 제공임을 필요할 때 짧게 밝힌다."""

FINAL_NUDGE = ("도구는 이제 그만 쓰고, 지금까지 받은 원문만으로 답해. "
               "확인하지 못한 부분은 「원문에서 확인하지 못했습니다」라고 밝히고, 사용자가 더 물어볼 방법(조항 번호 · 법령 이름)을 알려줘.")

def ask_steps(messages, max_tool_rounds=5):
    """GPT 가 도구를 쓰는 과정을 한 단계씩 알려 주는 함수
    - ("step", "🔧 ...")  : 도구를 쓰는 중
    - ("final", "답")      : 마지막 답
    도구 횟수를 다 쓰면 「한도 도달」로 끝내지 않고, 모은 원문으로 답을 받아 냅니다."""
    if not messages or messages[0].get("role") != "system":
        messages.insert(0, {"role": "system", "content": SYSTEM})
    done_calls = {}
    for _ in range(max_tool_rounds):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, **model_options())
        msg = resp.choices[0].message
        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content or ""})
            yield "final", msg.content or ""
            return

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            key = name + json.dumps(args, sort_keys=True, ensure_ascii=False)
            yield "step", f"🔧 {name}({json.dumps(args, ensure_ascii=False)})"
            if key in done_calls:
                out = {"note": "이미 같은 값으로 불렀습니다. 앞에서 받은 결과를 쓰세요."}
            else:
                try:
                    out = FUNCS[name](**args) if name in FUNCS else {"error": f"알 수 없는 도구: {name}"}
                except Exception as e:          # 도구 하나가 실패해도 대화는 이어 갑니다
                    out = {"error": f"{name} 실행 오류: {e}"}
                done_calls[key] = True
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, ensure_ascii=False)})

    # 도구 횟수를 다 썼을 때: 도구를 끄고(tool_choice="none") 모은 내용으로 답을 받습니다
    yield "step", "📝 모은 원문으로 답을 정리하는 중"
    messages.append({"role": "user", "content": FINAL_NUDGE})
    resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS,
                                          tool_choice="none", **model_options())
    text = resp.choices[0].message.content or "답을 만들지 못했습니다. 질문을 조금 좁혀(법령 이름 · 조항 번호) 다시 물어봐 주세요."
    messages.append({"role": "assistant", "content": text})
    yield "final", text

def ask(messages, verbose=True, max_tool_rounds=5):
    """한 번에 답만 받는 함수 (셀에서 테스트할 때 씁니다)"""
    for kind, text in ask_steps(messages, max_tool_rounds):
        if kind == "step" and verbose:
            print(text)
        if kind == "final":
            return text

# 테스트
history = [{"role": "user", "content": "GDPR에서 잊힐 권리는 어떤 경우에 행사할 수 있어?"}]
print(ask(history))


## 6. 채팅 UI (Gradio)

실행하면 아래에 채팅창이 뜨고, 공유용 링크(`gradio.live`)도 생깁니다.

In [ ]:
import gradio as gr, inspect

def to_text(content):
    # Gradio 6 은 대화 내용을 [{"type": "text", "text": ...}] 목록으로 주기도 합니다
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content or "")

def chat(message, ui_history):
    # 지금 Gradio(5 · 6)의 대화 기록은 {"role": ..., "content": ...} 형식입니다
    # (옛 코드처럼 [user, bot] 짝으로 풀면 오류가 나거나 엉뚱한 글이 들어갑니다)
    msgs = []
    for m in (ui_history or [])[-6:]:            # 앞 대화 3번까지만 기억 → 빠르고 싸게
        if isinstance(m, dict) and m.get("role") in ("user", "assistant"):
            msgs.append({"role": m["role"], "content": to_text(m.get("content"))})
    msgs.append({"role": "user", "content": to_text(message)})

    steps = []
    try:
        for kind, text in ask_steps(msgs):
            if kind == "step":
                steps.append(text)
                yield "⏳ 원문을 확인하는 중입니다…\n\n" + "\n".join(f"- `{s}`" for s in steps)
            else:
                yield text
    except Exception as e:
        s = str(e)
        if "insufficient_quota" in s:
            yield "⚠️ OpenAI 계정의 사용 한도(크레딧)가 없습니다. platform.openai.com 의 Billing 을 확인해 주세요."
        elif "rate_limit" in s or "429" in s:
            yield "⚠️ 요청이 너무 빨라 잠시 막혔습니다. 1분 뒤 다시 물어봐 주세요."
        else:
            yield f"오류가 발생했습니다: {e}"

kw = dict(
    fn=chat,
    title="🇪🇺 EUR-Lex 법령 챗봇",
    description="EU 법령을 한국어로 물어보세요. 답변은 EUR-Lex 원문을 근거로 합니다. 법령 이름과 조항 번호를 함께 적으면 더 빨리 답합니다.",
    examples=[
        "AI Act에서 고위험 AI 시스템은 어떻게 정의돼?",
        "2023년 이후 나온 배터리 관련 EU 규정 찾아줘",
        "DSA에서 초대형 온라인 플랫폼의 의무를 요약해줘",
    ],
)
if "type" in inspect.signature(gr.ChatInterface.__init__).parameters:   # Gradio 5 는 형식을 직접 지정
    kw["type"] = "messages"

gr.ChatInterface(**kw).launch(share=True, debug=True)


## 💡 확장 아이디어
- **RAG**: 자주 묻는 법령 본문을 미리 받아 조항 단위로 쪼갠 뒤 임베딩 → 벡터 검색으로 더 빠르고 정확하게
- **EuroVoc 주제 검색**: SPARQL에서 `cdm:work_is_about_concept_eurovoc`로 제목 대신 주제로 검색
- **판례 검색**: `doc_type="judgment"`로 EU 사법재판소 판결 검색
- **문제 해결**: SPARQL이 느리면 `limit`을 줄이거나 `year_from`을 지정하세요. 본문 요청이 막히면 잠시 후 다시 시도하세요(캐시가 있어 같은 문서는 한 번만 받습니다).